# hallucination-probes: train a detector

Uses saved features (output of `kaggle_extract.ipynb`, or the earlier `sala_qwen25_g14_6src_triviaqa_v1`
artifacts, which have the same layout). Edit the config cell: every entry in `RUNS` is one experiment
(`hprobes train` arguments); they run one after another and are compared at the end.

In [ ]:
# Clones the private repo using the GITHUB_TOKEN Kaggle secret (Add-ons -> Secrets -> attach it to this notebook).
# The token is passed to git only through an environment header, never in the command line or URL, so it cannot show
# up in tracebacks, logs or the saved clone.
import base64, os, subprocess, urllib.request
from kaggle_secrets import UserSecretsClient

REPO, REF = "brainardphilemon/hallucination-probes", "main"   # pin a commit hash for reproducible runs
DEST = "/kaggle/working/hallucination-probes"
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
       "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
subprocess.run(["rm", "-rf", DEST], check=True)
clone = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", DEST], env=env,
                       capture_output=True, text=True)
if clone.returncode:
    try:  # 401 = token invalid/expired; 404 = token cannot see this repo (wrong repository or missing Contents: read)
        request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
        status = urllib.request.urlopen(request).status
    except Exception as exc:
        status = getattr(exc, "code", type(exc).__name__)
    raise RuntimeError(f"git clone failed; GitHub API answered {status} for {REPO} with your token. 401: token invalid or "
                       "expired. 404: the token has no read access to this repository (create it with Resource owner "
                       "brainardphilemon, repository hallucination-probes, Contents: Read-only).") from None
del token, auth, env
subprocess.run(["git", "-C", DEST, "checkout", "-q", REF], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r /kaggle/working/hallucination-probes/requirements-kaggle.txt
!pip install -q --no-deps -e /kaggle/working/hallucination-probes

In [ ]:
# ---- config -------------------------------------------------------------------------------
ARTIFACT_NAME = "sala_qwen25_g14_6src_triviaqa_v1"   # attached input containing generation/ and bleurt/
SOURCES = ["tqa", "nq_open", "sciq", "popqa", "web_questions", "hotpotqa"]
TARGETS = ["triviaqa"]

RUNS = {
    # SALA without projections: pooled states -> per-layer PCA -> 4x128 MLP.
    # Each mlp run also saves runs/<name>/model.pt for `hprobes predict`.
    # This one reproduces kaggle.com/code/nvthaai/sala-g14-logsumexp-layerpca512-4x128 as it actually ran (PCA 256, BCE).
    "mlp_lse_pca256_bce": ["--method", "mlp", "--pooling", "logsumexp", "--pca-dim", "256",
                           "--pca-scope", "per-layer", "--loss", "bce"],
    # Model-size independent pipeline: any layer count -> 15 pooled layers, any hidden size -> 512 per layer (PCA is
    # fitted per model on its own features), so the MLP always sees 15 x 512.
    "mlp_15x512_bce": ["--method", "mlp", "--pooling", "logsumexp", "--pool-layers", "15", "--pca-dim", "512",
                       "--pca-scope", "per-layer"],
    # Same pipeline with PCA replaced by something that needs no fitting (any LLM, even one example):
    # "mlp_15x512_pca_self": ["--method", "mlp", "--pooling", "logsumexp", "--pool-layers", "15", "--pca-dim", "512", "--pca-fit", "self"],
    # "mlp_15x512_dct":      ["--method", "mlp", "--pooling", "logsumexp", "--pool-layers", "15", "--pca-dim", "512", "--pca-fit", "dct"],
    # "mlp_15x512_avg":      ["--method", "mlp", "--pooling", "logsumexp", "--pool-layers", "15", "--pca-dim", "512", "--pca-fit", "avg"],
    # "mlp_lse_pca512_focal": ["--method", "mlp", "--pooling", "logsumexp", "--pca-dim", "512",
    #                          "--pca-scope", "per-layer", "--loss", "focal"],
    # More examples (uncomment):
    # "mlp_max_bce":        ["--method", "mlp", "--pooling", "max"],
    # "mlp_raw_bce":        ["--method", "mlp", "--pooling", "none"],
    # "sala_lse_lodo":      ["--method", "sala", "--pooling", "logsumexp", "--lodo-dim"],
    # "sala_none_dim32":    ["--method", "sala", "--proj-dim", "32"],
    # "fusion_frozen_bge":  ["--method", "fusion", "--pooling", "logsumexp"],
    # "vlm_scratch":        ["--method", "vlm", "--pooling", "logsumexp"],
    # "poly_max":           ["--method", "polysemantic", "--pooling", "max"],
}
COMMON_ARGS = ["--seed", "42"]   # e.g. ["--epochs", "20", "--bleurt-threshold", "0.5"]

In [ ]:
from pathlib import Path
matches = [p for p in Path("/kaggle/input").rglob(ARTIFACT_NAME) if (p / "generation").is_dir()]
if len(matches) != 1:
    raise RuntimeError(f"Expected one attached artifact folder named {ARTIFACT_NAME}; found {matches}")
ARTIFACT_DIR = matches[0]
print("Features:", ARTIFACT_DIR)

In [ ]:
import json, subprocess, sys
import pandas as pd

rows = []
for name, run_args in RUNS.items():
    output_dir = Path("/kaggle/working/runs") / name
    command = [sys.executable, "-m", "hprobes.cli", "train", "--artifact-dir", str(ARTIFACT_DIR),
               "--output-dir", str(output_dir), "--sources", *SOURCES, "--targets", *TARGETS,
               *COMMON_ARGS, *run_args]
    print("\n=== ", name, "\n", " ".join(command), flush=True)
    subprocess.run(command, check=True)
    summary = json.loads((output_dir / "summary.json").read_text())
    for target, m in summary["target_metrics"].items():
        rows.append({"run": name, "target": target, "auroc": m.get("auroc"), "balanced_acc": m.get("balanced_accuracy"),
                     "accuracy": m.get("accuracy"), "precision": m.get("precision"), "recall": m.get("recall"),
                     "source_val_auroc": summary.get("best_source_val_auroc"),
                     "source_test_auroc": summary["source_test_metrics"].get("auroc")})
display(pd.DataFrame(rows).round(4))